# Baseline

#### Loading the dataset

In [1]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('autocorrect-aicc-round-1-2')

print("Path to competition files:", path)

Path to competition files: C:\Users\raian\.cache\kagglehub\competitions\autocorrect-aicc-round-1-2


In [2]:
import datasets
import pandas as pd
from tqdm.auto import tqdm

ds = datasets.load_dataset("csv", data_files=path+"/train.csv")
ds = ds["train"].train_test_split(seed=42)
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'misspell'],
        num_rows: 7928
    })
    test: Dataset({
        features: ['text', 'misspell'],
        num_rows: 2643
    })
})

In [3]:
train_ds = ds["train"]
val_ds = ds["test"]

# for later evaluation
val_ds_input = val_ds.select_columns("misspell")
val_ds_solution = val_ds["text"]

In [4]:
pd.DataFrame(val_ds)["text"].str.len().describe()

count    2643.000000
mean      628.893303
std       336.841864
min       102.000000
25%       379.000000
50%       581.000000
75%       819.500000
max      2291.000000
Name: text, dtype: float64

In [5]:
pd.DataFrame(val_ds)["misspell"].str.len().describe()

count    2643.000000
mean      628.893303
std       336.841864
min       102.000000
25%       379.000000
50%       581.000000
75%       819.500000
max      2291.000000
Name: misspell, dtype: float64

#### Preparing the data

In [6]:
from tokenizers import Tokenizer, models, pre_tokenizers, processors

tokenizer = Tokenizer(models.WordLevel(unk_token="<UNK>"))
tokenizer.pre_tokenizer = pre_tokenizers.Split("", "isolated")
tokenizer.enable_padding(pad_token="<PAD>")

In [7]:
trainer = tokenizer.model.get_trainer()
trainer.vocab_size = 1000
trainer.special_tokens = ["<PAD>", "<UNK>", "<SOS>", "<EOS>"]

In [8]:
def ds_iterator():
    for row in train_ds:
        yield row["text"]
        yield row["misspell"]

tokenizer.train_from_iterator(ds_iterator(), trainer=trainer)

In [9]:
tokenizer.post_processor = processors.TemplateProcessing(
    single="<SOS> $0 <EOS>",
    special_tokens=[("<SOS>", 2), ("<EOS>", 3)]
)

In [10]:
from transformers import PreTrainedTokenizerFast
tokenizer = PreTrainedTokenizerFast(tokenizer_object=tokenizer)
tokenizer.add_special_tokens({"pad_token": "<PAD>", "unk_token": "<UNK>", "cls_token": "<SOS>", "eos_token": "<EOS>"})
tokenizer

TokenizersBackend(name_or_path='', vocab_size=96, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<EOS>', 'unk_token': '<UNK>', 'pad_token': '<PAD>', 'cls_token': '<SOS>'}, added_tokens_decoder={
	0: AddedToken("<PAD>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<UNK>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<SOS>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<EOS>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [11]:
def tokenize_fn(examples):
    input_tokens = tokenizer(examples['misspell'], padding='max_length', truncation=True, max_length=768, return_tensors='pt')['input_ids']
    label_tokens = tokenizer(examples['text'], padding='max_length', truncation=True, max_length=768, return_tensors='pt')['input_ids']

    input_lengths = (input_tokens != tokenizer.pad_token_id).sum(dim=1)
    label_lengths = (label_tokens != tokenizer.pad_token_id).sum(dim=1)

    return {
        'input_ids': input_tokens,
        'labels': label_tokens,
        'input_lengths': input_lengths,
        'label_lengths': label_lengths,
    }

In [12]:
from torch.utils.data import DataLoader

train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=['text', 'misspell'])
train_ds.set_format("torch")
val_ds = val_ds.map(tokenize_fn, batched=True, remove_columns=['text', 'misspell'])
val_ds.set_format("torch")
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

#### Building the model

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class LSTMAutocorrect(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, lens):
        x = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(x, lens.cpu(), batch_first=True, enforce_sorted=False)
        packed_outputs, (_, _) = self.lstm(packed)
        x, _ = nn.utils.rnn.pad_packed_sequence(packed_outputs, batch_first=True)
        x = self.fc(x)
        return x
    

class BiGRU(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=2, dropout=0.2, pad_id=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=pad_id)
        self.bigru = nn.GRU(
            embed_size, hidden_size, num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(2 * hidden_size, vocab_size)

    def forward(self, x, lens=None, hidden=None):
        seq_len = x.size(1)
        x = self.embedding(x)
        if lens is None:
            out, hidden = self.bigru(x, hidden)
        else:
            packed = nn.utils.rnn.pack_padded_sequence(x, lens.cpu(), batch_first=True, enforce_sorted=False)
            out, hidden = self.bigru(packed, hidden)
            out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=seq_len)

        out = self.fc(self.dropout(out))
        return out


In [18]:
vocab_size = len(tokenizer)
embed_size = 64
hidden_size = 256
num_layers = 2
dropout = 0.2
num_epochs = 10
learning_rate = 2e-3
device = "cuda" if torch.cuda.is_available() else "cpu"

model = BiGRU(vocab_size, embed_size, hidden_size, num_layers, dropout,
              pad_id=tokenizer.pad_token_id).to(device)
# ignore_index stops the loss from being dominated by <PAD> positions
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-2)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=learning_rate, epochs=num_epochs, steps_per_epoch=len(train_loader)
)

print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

1,732,704 parameters


In [ ]:
use_amp = device == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
pad_id = tokenizer.pad_token_id
best_val = float("inf")


def run_epoch(loader, train: bool):
    model.train(train)
    total_loss, total_tokens, correct = 0.0, 0, 0

    with torch.set_grad_enabled(train):
        for batch in tqdm(loader, leave=False):
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            lengths = batch["input_lengths"]

            outputs = model(input_ids, lengths)
            loss = criterion(outputs.reshape(-1, vocab_size), labels.reshape(-1))

            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                # GRUs over ~768 steps: clip before the occasional exploding batch derails training
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

            # weight by real (non-pad) tokens so the epoch average matches the loss definition
            mask = labels != pad_id
            n = mask.sum().item()
            total_loss += loss.item() * n
            total_tokens += n
            correct += ((outputs.argmax(-1) == labels) & mask).sum().item()

    return total_loss / total_tokens, correct / total_tokens


for epoch in range(num_epochs):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)

    print(f"Epoch [{epoch+1}/{num_epochs}]  "
          f"train loss {train_loss:.4f} acc {train_acc:.4f}  |  "
          f"val loss {val_loss:.4f} acc {val_acc:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), "best_bigru.pt")
        print(f"  saved (best val loss {best_val:.4f})")

model.load_state_dict(torch.load("best_bigru.pt"))

  0%|          | 0/248 [00:00<?, ?it/s]

  0%|          | 0/83 [00:00<?, ?it/s]

Epoch [1/10]  train loss 0.0562 acc 352.3956  |  val loss 0.0182 acc 483.9342
  saved (best val loss 0.0182)


  0%|          | 0/248 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [16]:
def predict_sentence(model, tokenizer, input_ids, device="cpu"):
    """
    input_ids: tensor of shape (1, seq_len) with tokenized input
    """
    model.eval()
    with torch.no_grad():
        input_ids = input_ids.to(device)
        outputs = model(input_ids, torch.tensor([input_ids.size(1)], dtype=torch.int64))  # (1, seq_len, vocab_size)
        predictions = outputs.argmax(dim=-1)  # (1, seq_len)

    # Convert ids back to tokens / string
    predicted_tokens = tokenizer.convert_ids_to_tokens(predictions[0][1:-1].tolist())
    return "".join(predicted_tokens)

# Suppose you have a test sentence
test_text = " Good morning rveryone. How are you. "  # misspelled input

# Tokenize and convert to tensor
input_ids = tokenizer.encode(test_text, return_tensors="pt")

# Get prediction
corrected = predict_sentence(model, tokenizer, input_ids, device=device)
print("Input:", test_text)
print("Corrected:", corrected)

Input:  Good morning rveryone. How are you. 
Corrected:  Good Morningreverine . How areyyou. 


# Evaluation

#### Predicting on Test

In [17]:
test_ds = datasets.load_dataset("csv", data_files="/kaggle/input/autocorrect-aicc-round-1-2/test.csv")
# test_ds = val_ds_input # for validation
test_ds = test_ds["train"] # for test
test_ds

FileNotFoundError: Unable to find '/kaggle/input/autocorrect-aicc-round-1-2/test.csv'

In [ ]:
def tokenize_test_fn(examples):
    input_tokens = tokenizer(examples['misspell'], padding='max_length', truncation=True, max_length=4096, return_tensors='pt')['input_ids']
    input_lengths = (input_tokens != tokenizer.pad_token_id).sum(dim=1)

    return {
        'input_ids': input_tokens,
        'input_lengths': input_lengths
    }

test_ds = test_ds.map(tokenize_test_fn, batched=True, remove_columns=['misspell'])
test_ds.set_format("torch")

test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

In [ ]:
def wrap_model_prediction(model, batch, device):
    input_ids = batch['input_ids'].to(device)
    lengths = batch['input_lengths'].to(device)

    return model(input_ids, lengths)

**Note: For this task, the requirement for the evaluation to finish in less than 250 seconds will be based on this block of code below. DO NOT MODIFY ANY CODE - use the wrapper function above for any changes due to your model architecture.** Please don't try anything that goes against the spirit of this challenge...

In [ ]:
%%time
device = "cpu" # change to cuda for validation
preds_all = []

model.eval()
model.to(device)
with torch.no_grad():
    for batch in tqdm(test_loader):
        outputs = wrap_model_prediction(model, batch, device)
        predictions = outputs.argmax(dim=-1)
        preds_all.append(predictions.cpu())

In [ ]:
results = [] # convert tokens back to string, excluded from timed evaluation as this takes quite a while
for pred in tqdm(preds_all):
    results += [ "".join(tokenizer.convert_ids_to_tokens(x, skip_special_tokens=True)) for x in pred ]

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df = df.reset_index()
df.columns = ["id", "corrected"]
df.to_csv("/kaggle/working/test_submission.csv", index=False)

print("test_submission.csv generated!")

#### Code for Evaluation (for Validation)
Unfortunately, due to environment restrictions, evaluation on the server uses the Python-based implementation provided by `torchmetrics` instead of the much faster implementation provided by `jiwer`. Hence, results may vary slightly, and expect server-based eval to take about 10 minutes.

In [ ]:
#! pip install jiwer evaluate --quiet
#import evaluate

#cer = evaluate.load("cer")

In [ ]:
#import pandas as pd

#submission = pd.read_csv("test_submission.csv")
# solution = pd.DataFrame(pd.Series(val_ds_solution)) # for validation
#solution = pd.read_csv("test_sol.csv") # for testing

In [ ]:
#cer.compute(
#    predictions=submission["corrected"],
#    references=solution.iloc[:, 0]
#)